### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="bank_customer_churn",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/gauravtopre/bank-customer-churn-dataset",
    download_description="""
We download the data from Kaggle to a .csv file in a predefined folder.
    
mkdir -p local-data-warehouse/bank_customer_churn && cd local-data-warehouse/bank_customer_churn && kaggle datasets download gauravtopre/bank-customer-churn-dataset && unzip bank-customer-churn-dataset.zip && rm bank-customer-churn-dataset.zip && cd ../../
""",
    # References
    academic_reference_bibtex="""@misc{Topre2022BankCustomerChurn,
  title={Bank Customer Churn Dataset},
  author={Gaurav Topre},
  year={2022},
  publisher={Kaggle},
  url={https://www.kaggle.com/datasets/gauravtopre/bank-customer-churn-dataset}
}
""",
    academic_reference_bibtex_key="Topre2022BankCustomerChurn",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We remove the customer_id column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="churn",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="churn",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "Bank Customer Churn Prediction.csv")

df["churn"] = df["churn"].map({1: "Yes", 0: "No"})

cat_cols = ["churn", "active_member", "country", "gender", "credit_card"]
df[cat_cols] = df[cat_cols].astype("category")

df = df.drop(columns=["customer_id"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()